# LLMs in Practice: Prompting, RAG & LoRA

**CS 4771/5771 — Python for Machine Learning · Module 8: Advanced Topics**

This notebook accompanies Session 28 (*LLMs in Practice*). It is designed to run on the
**free Google Colab tier** (a T4 GPU helps but is not required — every model here is small).

We will:

1. Prompt a small open-weight LLM (zero-shot vs. few-shot)
2. Build a **mini-RAG** system: embeddings + cosine-similarity retrieval + prompt augmentation
3. Sketch **parameter-efficient fine-tuning** with LoRA (via the `peft` library)
4. Close with a checklist for verifying AI-generated output

## 0. Setup

Uncomment and run the cell below once per Colab session. (On Colab, `transformers` and
`torch` are usually preinstalled; `sentence-transformers` and `peft` are not.)

In [ ]:
# %pip install -q transformers sentence-transformers datasets peft accelerate

import torch
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {device}")

## 1. Prompting a small open model

We use **Qwen2.5-0.5B-Instruct** — a 0.5B-parameter instruction-tuned model that downloads
in about a gigabyte and runs even on the Colab CPU (slowly) or T4 GPU (comfortably).

> **Substitutions:** any small instruct model works the same way, e.g.
> `HuggingFaceTB/SmolLM2-360M-Instruct` or `TinyLlama/TinyLlama-1.1B-Chat-v1.0`.
> Expect small models to make mistakes — that is part of the lesson.

In [ ]:
from transformers import pipeline

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

generator = pipeline(
    "text-generation",
    model=MODEL_NAME,
    device_map="auto",          # uses the GPU if Colab gave you one
    torch_dtype="auto",
)


def chat(prompt, max_new_tokens=80):
    """Send a single-turn chat prompt and return the assistant's reply."""
    messages = [{"role": "user", "content": prompt}]
    out = generator(messages, max_new_tokens=max_new_tokens, do_sample=False)
    return out[0]["generated_text"][-1]["content"].strip()


print(chat("In one sentence, what does a diffusion model learn to do?"))

### Zero-shot vs. few-shot

The task: classify a course-feedback comment into exactly one of three labels.
First we just *describe* the task (zero-shot); then we *show* the pattern with a few
worked examples (few-shot / in-context learning). Watch how the examples pin down both
the label set and the output format.

In [ ]:
comment = "The homework took forever, but I finally understand backpropagation."

zero_shot = f"""Classify the following course feedback as POSITIVE, NEGATIVE, or MIXED.
Reply with exactly one word.

Feedback: {comment}"""

print("ZERO-SHOT:", chat(zero_shot, max_new_tokens=10))

In [ ]:
few_shot = f"""Classify course feedback as POSITIVE, NEGATIVE, or MIXED.
Reply with exactly one word.

Feedback: The lectures were clear and the pace was perfect.
Label: POSITIVE

Feedback: I could never hear the instructor and the slides were outdated.
Label: NEGATIVE

Feedback: Great projects, though the grading felt slow.
Label: MIXED

Feedback: {comment}
Label:"""

print("FEW-SHOT:", chat(few_shot, max_new_tokens=10))

**Try it:** change `comment` to something sarcastic ("Oh great, *another* proctored exam")
and compare the two again. Few-shot prompting usually stabilizes the output *format*
immediately; whether it fixes the *judgment* depends on the model.

Two more techniques to experiment with:

- **Chain-of-thought:** append "Think step by step before giving the label." (allow more
  `max_new_tokens`).
- **Structured output:** ask for `{"label": ..., "confidence": ...}` as JSON and parse it
  with `json.loads` — and notice how often a small model breaks the format. Production
  systems validate and retry.

## 2. Mini-RAG: retrieval-augmented generation

The model above has never seen *our course's* logistics, and its knowledge stops at its
training cutoff. RAG fixes this: **retrieve** relevant documents at question time and put
them **in the prompt**.

Our "document store" is ten strings; the real-world version chunks PDFs and wikis, but the
pipeline shape is identical: **embed docs → embed query → similarity search → augment prompt**.

In [ ]:
corpus = [
    "CS 4771/5771 final project proposals are due Friday, November 13, 2026 on Canvas.",
    "Office hours for CS 4771/5771 are Tuesdays and Thursdays, 2:00-3:30pm, in JEB 321.",
    "Late homework in CS 4771/5771 loses 10% per day and is not accepted after 3 days.",
    "The CS 4771/5771 midterm exam covers modules 1 through 5 and is closed-book.",
    "Students may use AI assistants on homework only if they disclose the usage and can explain every line submitted.",
    "The final project presentation counts for 15% of the CS 4771/5771 course grade.",
    "CS 4771/5771 uses Python 3.12, PyTorch, and scikit-learn; environment setup is in Module 0.",
    "Graduate students in CS 5771 must additionally submit a 4-page project report in NeurIPS format.",
    "The last day to drop CS 4771/5771 without a W grade is September 8, 2026.",
    "Regrade requests must be submitted within one week of a grade being posted.",
]

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")   # 384-dim embeddings, ~80 MB

doc_embeddings = embedder.encode(corpus, normalize_embeddings=True)
print("Embedding matrix shape:", doc_embeddings.shape)   # (10, 384)

In [ ]:
def retrieve(query, k=3):
    """Return the top-k most similar documents to the query (cosine similarity)."""
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = doc_embeddings @ q          # normalized vectors -> dot product = cosine sim
    top = np.argsort(scores)[::-1][:k]
    return [(float(scores[i]), corpus[i]) for i in top]


for score, doc in retrieve("When is the project proposal due?"):
    print(f"{score:.3f}  {doc}")

In [ ]:
question = "Am I allowed to use AI assistants on my homework?"

# Without retrieval, the model can only guess (or hallucinate) our course policy:
print("NO CONTEXT:", chat(question, max_new_tokens=60), "\n")

# With retrieval, we ground the answer in the course documents:
context = "\n".join(doc for _, doc in retrieve(question, k=3))
rag_prompt = f"""Answer the question using ONLY the context below.
If the context does not contain the answer, say "I don't know."

Context:
{context}

Question: {question}"""

print("WITH RAG:", chat(rag_prompt, max_new_tokens=80))

**Things to notice:**

- The retriever and the generator are *separate, swappable components*. Scale the corpus to
  millions of chunks by swapping the numpy search for a vector index (FAISS, Chroma, pgvector) —
  nothing else changes.
- The instruction *"say I don't know if the context is insufficient"* is the cheapest
  hallucination guard you will ever write. Test it: ask "What textbook does the course use?"
  (which the corpus does not answer).

## 3. Parameter-efficient fine-tuning with LoRA

RAG injects *knowledge*; fine-tuning changes *behavior*. Full fine-tuning updates every
weight — infeasible on free hardware. **LoRA** freezes the base model and learns a low-rank
update ($W x + BAx$ with rank $r \ll d$) on selected layers, shrinking the trainable
parameter count by orders of magnitude.

We demonstrate on **distilgpt2** (82M parameters) so the optional training cell below is
CPU-tolerable.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

BASE = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(BASE)
tokenizer.pad_token = tokenizer.eos_token          # GPT-2 has no pad token by default

base_model = AutoModelForCausalLM.from_pretrained(BASE)

lora_config = LoraConfig(
    r=8,                        # rank of the adapter matrices A (d x r) and B (r x d)
    lora_alpha=16,              # scaling factor applied to the adapter output
    lora_dropout=0.05,
    target_modules=["c_attn"],  # GPT-2's fused attention projection layer
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
# Expect well under 1% trainable — this is why LoRA fits on free hardware.

### Optional: a few training steps

This stub teaches the model a house style from six examples — obviously too little data to
matter, but it exercises the *entire* LoRA training loop in under a minute, even on CPU.
Real runs use `transformers.Trainer` (or `trl`'s `SFTTrainer`) with a proper dataset
(e.g., loaded via the `datasets` library); only the adapter weights ever receive gradients.

**This cell is optional — skip it if you are short on time.**

In [ ]:
# --- OPTIONAL TRAINING CELL ---
tiny_dataset = [
    "Q: What is overfitting? A: In ML terms: memorizing noise instead of learning signal.",
    "Q: What is a tensor? A: In ML terms: an n-dimensional array with autograd superpowers.",
    "Q: What is a loss function? A: In ML terms: a score for how wrong the model is.",
    "Q: What is gradient descent? A: In ML terms: rolling downhill on the loss surface.",
    "Q: What is regularization? A: In ML terms: a penalty that keeps the model humble.",
    "Q: What is an embedding? A: In ML terms: a learned vector that encodes meaning.",
]

batch = tokenizer(tiny_dataset, return_tensors="pt", padding=True)
batch["labels"] = batch["input_ids"].clone()

model.train()
optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad), lr=2e-4
)

for step in range(10):                       # a real run would take thousands of steps
    outputs = model(**batch)
    outputs.loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    print(f"step {step:2d}  loss = {outputs.loss.item():.3f}")

# The trained adapter alone is tiny and can be saved/shared separately from the base model:
# model.save_pretrained("distilgpt2-lora-adapter")   # a few MB, not 300+ MB

**Key takeaways:** the loss goes down while >99% of the model stays frozen; the saved
artifact is a few-megabyte adapter, not a full model copy; and you can keep one base model
with a different adapter per task. QLoRA extends the same idea to 4-bit quantized bases,
which is how 7B+ models get fine-tuned on a single free Colab GPU.

## 4. Verifying AI-generated output: a checklist

LLM output is fluent whether or not it is correct. Before you trust — or submit — anything
a model produced, walk this list (it mirrors the course AI-use policy: *you* are
accountable for everything you turn in):

- [ ] **Can I explain every line?** If not, I don't submit it.
- [ ] **Did I run it?** On a case where I already know the correct answer, plus at least one
  edge case (empty input, wrong shape, boundary value).
- [ ] **Do the APIs actually exist?** Models invent plausible function names and signatures —
  check the real documentation.
- [ ] **Are factual claims grounded?** For RAG systems: is every statement in the answer
  supported by a retrieved document?
- [ ] **Is the format valid?** Structured output (JSON, code) gets parsed/validated
  programmatically, never eyeballed.
- [ ] **Would it survive a re-run?** Re-ask the same question; confident answers that change
  on every run deserve suspicion.
- [ ] **Did I build an eval set?** Even 20 examples, re-run on every prompt or model change,
  beats vibes.

**Further reading:** the course page
[LLMs in Practice](../../modules/advanced-topics/llms-in-practice.md) covers RAG vs.
fine-tuning trade-offs and prompt versioning in more depth.